In [0]:
SELECT *
FROM bright.bright_tv.bright_tv_dataset_viewership
LIMIT 1;

--- Note: The UserID is the foreign ket to this table, but a primary key to the User_Profile.

--------------------------------------------
-- How big is the dataset?

SELECT COUNT(*) AS number_of_rows,
       COUNT(COALESCE(UserID0,userid4)) AS active_subs     ---10000 (Used COALESCE because it can scan multiple rows)
FROM bright.bright_tv.bright_tv_dataset_viewership; ---10000

--- You cannot COUNT(DISTINCTUserID) distinctly because its a foreign key.
--- If you count distinct, you would want to find active subscribers, that are watching.

--- How many active users?

SELECT COUNT(DISTINCT COALESCE(UserID0,userid4)) AS active_users
FROM bright.bright_tv.bright_tv_dataset_viewership; --- There are 4386 active users

--- IS THERE A ROW WHERE THE UserID column is null?
SELECT COUNT(*) AS number_of_rows,
       COUNT(COALESCE(UserID0,userid4)) AS active_subs    
FROM bright.bright_tv.bright_tv_dataset_viewership
WHERE UserID0 IS NULL; -- 0, which means the dataset does nit have a null value on UserID0


SELECT COUNT(*) AS number_of_rows,
       COUNT(COALESCE(UserID0,userid4)) AS active_subs    
FROM bright.bright_tv.bright_tv_dataset_viewership
WHERE userid4 IS NULL;  -- 0, which means the dataset does nit have a null value on UserID4


---- Channel Analysis

SELECT DISTINCT Channel2
FROM bright.bright_tv.bright_tv_dataset_viewership
LIMIT 1;--- 'Sawsee, SuperSport Live Events, Live on SuperSport' is repeated. Needs to be sorted- use CASE STATEMENT

SELECT DISTINCT 
    CASE
        WHEN Channel2 IN ('Sawsee','Sawsee') THEN 'Sawsee'  --- This is to combine the two channels into one
        WHEN Channel2 = 'SuperSport Live Events' THEN 'Live Events'
        WHEN Channel2 IN('SuperSport Live Events') THEN 'Live Events'
        WHEN Channel2 = 'Live on SuperSport' THEN 'Live Events'
    ELSE Channel2
    END AS TV_Channel
FROM bright.bright_tv.bright_tv_dataset_viewership; --- I keep getting a 20 rows as the outcome.
--- Let me fix the code.

SELECT DISTINCT 
    CASE
        WHEN Channel2 IN ('SawSee','SawSee') THEN 'Sawsee'  --- This is to combine the two channels into one
        WHEN Channel2 IN('SuperSport Live Events', 'Live on SuperSport','Supersport Live Events','DStv Events 1') THEN 'Live Events'
       
    ELSE Channel2
    END AS TV_Channel
FROM bright.bright_tv.bright_tv_dataset_viewership; --- Another reason I was getting 'errow' was the spelling of 'sawsee'. Values in the column are case sensitive! 
-- The final outcome has 17 rows.

------------------------------
-- RecordDate2 Analysis

WITH BASE AS (
    SELECT COALESCE(UserID0,userid4)AS user_id
    FROM bright.bright_tv.bright_tv_dataset_viewership
),

Processing AS(
SELECT
    COALESCE(UserID0,userid4)AS user_id,
    HOUR(RecordDate2) AS hour_of_day,
    TO_CHAR(RecordDate2,'yyyyMM') AS month_id,
    TO_DATE(RecordDate2) AS watch_date,
   -- TIME(RecordDate2) AS watch_time,
    DAY(RecordDate2) AS day_of_month,
    DAYNAME(RecordDate2) AS day_name,
    MONTHNAME(RecordDate2) AS month_name,
 
CASE 
    WHEN day_name IN ('Sat','Sun') THEN 'weekend'
    ELSE 'weekday'
END AS day_category,

date_format(`Duration 2`,'HH:mm:ss') AS duration,
HOUR(RecordDate2) AS hour_of_day,
date_format(RecordDate2,'HH:mm:ss') AS watch_time,

CASE
    WHEN Channel2 IN ('SawSee','SawSee') THEN 'Sawsee'  
    WHEN Channel2 IN('SuperSport Live Events', 'Live on SuperSport','Supersport Live Events','DStv Events 1') THEN 'Live Events'
       
    ELSE Channel2
    END AS TV_Channel


FROM bright.bright_tv.bright_tv_dataset_viewership

)
